# Dependency Resolution Lab

This notebook edits one local `pyproject.toml` beside the notebook and then uses `uv` to lock, validate, and sync dependencies.


## Fast mental model

- Edit `pyproject.toml` when dependency intent changes.
- Run `uv lock` to resolve and write `uv.lock`.
- Run `uv sync` to make `.venv` match the lockfile.
- Run `uv sync --frozen` when the lockfile must not change.
- Run `uv lock --check` to catch lockfile drift.

## Create the project file

The project file lives directly beside this notebook. One dependency is exact-pinned so the upgrade story is easy to see.

Remove any existing `pyproject.toml`.

In [ ]:
%%bash
set -euxo pipefail
rm -f pyproject.toml uv.lock

: 

In [ ]:
%%bash
cat > pyproject.toml <<'EOF'
[project]
name = "uv-resolution-lab"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
  "fastapi>=0.95,<0.96",
  "httpx==0.24.1",
  "rich>=13.7,<14"
]
EOF

## Show the starting project file

Always inspect the file you just wrote before resolving anything.

In [ ]:
%%bash
set -euxo pipefail
cat pyproject.toml

## Resolve dependencies into a lockfile

Use this after a manual edit to `pyproject.toml`. `uv lock` changes the lockfile, not the environment.

In [ ]:
%%bash
set -euxo pipefail
# Resolve the declared dependencies
uv lock

## Create the environment from the lockfile

Use this when you want the local environment to match the resolved dependency graph.

In [ ]:
%%bash
set -euxo pipefail
# Install from the lockfile
uv sync

## Edit a single pinned dependency

A direct file edit is the clearest way to change an exact pin such as `httpx==0.24.1`.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.24.1/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

## Refresh the lockfile after the pin change

After changing the pin, resolve again so `uv.lock` records the new exact version.

In [ ]:
%%bash
set -euxo pipefail
# Re-resolve after editing the pin
uv lock

## Install the new pin

Lock first, then sync. That order avoids mutating `.venv` from stale dependency state.

In [ ]:
%%bash
set -euxo pipefail
# Install the newly locked versions
uv sync

## Try a broken pin

A bad exact pin should fail during resolution. That is the safe failure point.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.27.2/httpx==999.0.0/' pyproject.toml
grep -n 'httpx' pyproject.toml

## Observe the resolution failure

Use `uv lock` here to confirm that the edited constraint cannot be satisfied.

In [ ]:
%%bash
set -euxo pipefail
# Resolve the edited constraints
uv lock || true

## Restore a valid pin

Repair the exact version in the file, then resolve again.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==999.0.0/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

## Rebuild the repaired lockfile

Run this after fixing a bad manual edit.

In [ ]:
%%bash
set -euxo pipefail
# Resolve the repaired constraints
uv lock

## Check for lockfile drift

Use this in CI or before commit. It verifies that `pyproject.toml` and `uv.lock` still agree.

### Create lockfile drift

Make a small manual edit in `pyproject.toml` without rebuilding `uv.lock`.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.8,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

In [ ]:
%%bash
set -euxo pipefail
# Verify that the lockfile no longer matches pyproject.toml
uv lock --check || true

### Revert the manual change

Restore the original dependency line, then confirm that the lockfile is valid again.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.8,<14/rich>=13.7,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

In [ ]:
%%bash
set -euxo pipefail
# Verify that the lockfile matches pyproject.toml again
uv lock --check

## Install strictly from the existing lockfile

This section contrasts two cases: a stale lockfile that makes `uv sync --frozen` fail and a refreshed lockfile that lets it succeed.

### Stale lockfile blocks installation

Edit `pyproject.toml` without rebuilding `uv.lock`. In that state, frozen sync should fail.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.9,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

In [ ]:
%%bash
set -euxo pipefail
# Try to install from a stale lockfile
uv sync --frozen || true

### Rebuild the lockfile first

Once `uv.lock` is refreshed, frozen sync can install successfully again.

In [ ]:
%%bash
set -euxo pipefail
# Refresh the lockfile after editing pyproject.toml
uv lock

In [ ]:
%%bash
set -euxo pipefail
# Install strictly from the refreshed lockfile
uv sync --frozen

## Upgrade a single dependency

This section contrasts two cases: an exact constraint that prevents movement and a loose constraint that allows a targeted upgrade.

### Fixed constraint prevents changes

Pin the dependency to one exact version first. In that state, the upgrade command should have no effect.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.95,<0.96/fastapi==0.95.2/' pyproject.toml
grep -n 'fastapi' pyproject.toml

In [ ]:
%%bash
set -euxo pipefail
# Try to upgrade the exact-pinned dependency
uv lock --upgrade-package fastapi

### Loose constraint allows changes

Relax the version constraint, then the targeted upgrade command can refresh that dependency within the allowed range.

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi==0.95.2/fastapi>=0.95,<0.96/' pyproject.toml
grep -n 'fastapi' pyproject.toml

In [ ]:
%%bash
set -euxo pipefail
# Upgrade the dependency after loosening its constraint
uv lock --upgrade-package fastapi

## Refresh all ranged dependencies

Use this for a broader maintenance pass when you want the newest versions allowed by your declared ranges.

In [ ]:
%%bash
set -euxo pipefail
# Upgrade all dependencies allowed by the declared ranges
uv lock --upgrade